# EDA and data cleaning for the PPR csv data

## Imports

In [ ]:
import pandas as pd
from pathlib import Path
import re
from deepparse.parser import AddressParser
import numpy as np

## read raw data

In [ ]:
# Path to the file we downloaded in Iteration 1
raw_data_path = Path("../data/raw/PPR-ALL.csv")

# Note: The PPR file uses 'latin-1' or 'cp1252' encoding, not standard 'utf-8'
raw_df = pd.read_csv(raw_data_path, encoding='cp1252')

## read a sample from the raw data

In [ ]:
print(f"Dataset contains {len(raw_df):,} rows.")
raw_df.head()

## create deep copy of the raw data

In [ ]:
df = raw_df.copy(deep=True)

## clean columns names

In [ ]:
df.columns = (
    df.columns
    .str.replace(r'[^\w\s]', '', regex=True) # Removes (€) and other symbols
    .str.strip()                             # Removes leading/trailing spaces
    .str.replace(' ', '_')                   # Replaces middle spaces with underscores
    .str.lower()                             # Makes everything lowercase
)
print(df.columns)

## Price cleaning and enrich

### clean price column from non numeric symbols

#### create a function for cleaning the price string

In [ ]:
def clean_currency(price_str):
    if pd.isna(price_str):
        return None
    # remove everything that isn't a digit or descimal point
    clean_str = re.sub(r'[^\d.]','',str(price_str))
    # convert clean_str to floot which handles the .00 then to int
    try:
        return int(float(clean_str))
    except ValueError:
        return None

#### use the function to clean

In [ ]:
df['price_clean'] = df['price'].apply(clean_currency)
print(f"Average Price in euros: {df['price_clean'].mean()}")

### create boolean column for vat exclusive

In [ ]:
df['is_vat_exclusive'] = df['vat_exclusive'].str.contains('Yes',case=False, na=False)

## create a smaller sample dataset for fast debugging and development

#### new folder so raw data are safe

In [ ]:
# Define your paths
processed_dir = Path("../data/processed")

# Create the folder if it doesn't exist
processed_dir.mkdir(parents=True, exist_ok=True)

#### filter county dublin only

In [ ]:
# 1. Filter for Dublin (Case-insensitive just in case)
dublin_df = df[df['county'].str.contains('Dublin', case=False, na=False)].copy()

# 2. Save to new folder
output_path = processed_dir / "dublin_sample.csv"
dublin_df.to_csv(output_path, index=False)

print(f"Success! Saved {len(dublin_df):,} rows to {output_path}")

## Deepparse to parse address

### use deepparse on the dublin sample dataset

In [ ]:
df_dublin = pd.read_csv("../data/processed/dublin_sample.csv")
#Initialize the parser (the first time takes a moment to download the model)
# We use 'fasttext' because it's lightweight and runs great on a Mac
address_parser = AddressParser(model_type="bpemb", device="cpu")
#Test on a single messy Dublin address
test_address = df_dublin['address'].iloc[0]
parsed = address_parser(test_address)

print(f"Original: {test_address}")
print(f"Parsed: {parsed}")


### function to convert address string to dictionary with address components

In [ ]:
def parse_address_to_cols(address_str):
    try:
        return address_parser(address_str)
    except Exception as e:
        return {"error": str(e)}


In [ ]:
def fast_parse(address_list):
    # num_workers=2 or 4 uses multiple CPU cores
    # batch_size=256 processes many addresses at once
    parsed_results = address_parser(address_list, batch_size=256)
    return [obj.to_dict() for obj in parsed_results]

In [ ]:
df_to_parse = df_dublin
# 2. Split into batches
batches = np.array_split(df_to_parse['address'].tolist(), 30)
all_results = []
print(f"Starting parsing of {len(df_to_parse)} rows...")

for i, batch in enumerate(batches):
    print(f"Processing batch {i+1}/30...")
    all_results.extend(fast_parse(batch.tolist()))

# 3. Create the final DataFrame
df_parsed = pd.DataFrame(all_results)
df_final = pd.concat([df_to_parse.reset_index(drop=True), df_parsed], axis=1)

print("Done! Here is a sample:")
print(df_final[['Address', 'StreetNumber', 'StreetName']].head())

In [ ]:
print("Done! Here is a sample:")
print(df_final[['address', 'StreetNumber', 'StreetName']].head())

In [ ]:
# Save to a compressed CSV to save space, or a standard CSV
df_final.to_csv("../data/processed/ppr_dublin_parsed.csv", index=False)
print("File saved successfully!")

In [ ]:
#Check for common Irish address components
# See how many times it found a 'Municipality' (usually the County/Town)
print(df_final['Municipality'].value_counts().head(10))

# Check for rows where it couldn't find a StreetName
null_streets = df_final['StreetName'].isnull().sum()
print(f"Rows without a Street Name: {null_streets}")

In [ ]:
# Convert to lowercase, remove 'co ', and strip 'dublin' from the end 
# so 'lucan dublin' becomes just 'lucan'
df_final['Municipality_Clean'] = (
    df_final['Municipality']
    .str.lower()
    .str.replace('co ', '', regex=False)
    .str.replace(' dublin', '', regex=False)
    .str.strip()
)

print(df_final['Municipality_Clean'].value_counts().head(10))

In [ ]:
# See what these 383 rows actually look like
missing_streets = df_final[df_final['StreetName'].isnull()]
print(missing_streets['address'].head(10))